# Vector Embeddings

**Dense numeric vectors that place meaning in geometry** — the representation that lets a model turn words, sentences, images, or users into points in space where *distance ≈ dissimilarity*.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**What:** A vector embedding is a fixed-length array of floats — `[0.12, -0.83, ..., 0.04]`, typically 256–4096 dimensions — produced by a model so that **semantically similar inputs land close together** and unrelated ones land far apart. The model learns to fold meaning into geometry: direction and distance carry information that the raw token IDs or pixels never did.

**The problem it solves:** Computers compare text by exact string match — "car" and "automobile" are as different as "car" and "banana". Embeddings replace symbolic identity with **continuous similarity**. Once your data is a set of vectors you can rank by *closeness*, cluster, deduplicate, recommend, and retrieve by meaning instead of keywords. They are the substrate underneath semantic search, RAG, recommendation, clustering, and classification.

**Reach for embeddings when:**
- You need **semantic** search/matching, not lexical — "how do I reset my password" should match a doc titled "account recovery". See [[semantic-search]].
- You're building **RAG** and need to retrieve relevant context for an LLM. See [[rag]], [[vector-db-comparison]].
- You want to **cluster, deduplicate, classify, or recommend** over text/images by similarity.

**Don't reach for them when:**
- You need **exact / keyword** matching (IDs, SKUs, legal citations) — BM25 / a SQL `LIKE` is simpler and exact. Often you want **both** (hybrid search).
- The corpus is tiny and human-readable — embeddings add a model dependency you may not need.
- You need an **interpretable** decision boundary — individual dimensions are not human-meaningful.

## 2. Mental Model

Picture a giant **map of meaning**. Every sentence is a pin on it. The embedding model is the cartographer: it places "the cat sat on the mat" and "a feline rested on the rug" almost on top of each other, and "quarterly revenue grew 12%" on a different continent.

```
        finance                         pets
   . revenue                       . cat on the mat
   . earnings call          . feline on the rug
   . stock split                  . dog fetches ball
                          . puppy plays
        (distance = dissimilarity; direction can encode relationships)
```

Two things make this useful:

1. **Distance encodes meaning.** Nearest neighbors = most similar items. Search becomes "find the closest pins."
2. **Directions can encode relations.** The classic toy result `vec("king") - vec("man") + vec("woman") ≈ vec("queen")` shows a "royalty" and a "gender" direction the model discovered on its own. Real LLM-era embeddings are messier than that clean analogy, but the intuition holds: *offsets in the space mean something*.

You almost never look at the numbers. You compute a **similarity** between two vectors (usually cosine) and rank. The embedding model and the vector index are two halves of one pipeline: the model makes the pins, the index ([[faiss]], [[pinecone]], …) finds the nearest ones fast.

## 3. Key Concepts

- **Dimension (`d`)** — the length of the vector (e.g. 384 for `all-MiniLM-L6-v2`, 1536 for OpenAI `text-embedding-3-small`). Higher `d` can hold more nuance but costs memory and compute; it is **not** automatically "better".
- **Cosine similarity** — the angle between two vectors, `cos θ = (a·b)/(‖a‖‖b‖)`, ranging −1…1. The default similarity for text embeddings because it ignores magnitude and compares **direction** (meaning). Cosine **distance** = `1 − cosine similarity`.
- **Dot product vs cosine** — dot product also factors in vector length. On **L2-normalized** vectors (unit length) dot product *equals* cosine — which is why pipelines normalize once and then use the cheaper dot product. See [[faiss]].
- **Euclidean (L2) distance** — straight-line distance. On normalized vectors it ranks identically to cosine, so the choice rarely changes results once you normalize.
- **Normalization** — scaling a vector to length 1. Makes cosine/dot/L2 agree and keeps magnitudes from dominating. Most sentence-embedding models recommend it.
- **Dense vs sparse** — embeddings are **dense** (every dimension has a value, meaning is distributed). Contrast with **sparse** lexical vectors (TF-IDF/BM25), which are mostly zeros and tied to exact words. Hybrid search blends both.
- **Embedding model** — what produces vectors: bi-encoders like Sentence-Transformers / `text-embedding-3`, or a pooled hidden layer of an LLM. Different models live in **different, incompatible spaces** — never mix vectors from two models in one index.
- **Pooling** — how token vectors become one sentence vector (mean-pooling, [CLS] token, last-token). An implementation detail of the model, but it's why "sentence embedding" ≠ "average of word embeddings" in general.
- **Anisotropy / the similarity floor** — raw transformer embeddings often cluster in a narrow cone, so *everything* looks ~0.3–0.9 similar. Judge **relative** ranking, not absolute scores, and calibrate per model.

## 4. Setup

The always-on examples below use only **numpy** so they run on any CPU in milliseconds — we hand-build tiny vectors to expose the math. The final cell *optionally* uses `sentence-transformers` to make **real** embeddings; it's gated so the notebook still runs end-to-end without it.

```bash
pip install numpy
# optional, for the real-model cell (downloads a ~80 MB model on first use):
pip install sentence-transformers
```

There is no API key or GPU needed for anything here.

In [1]:
import numpy as np

print("numpy", np.__version__)


def cosine(a, b):
    """Cosine similarity between two 1-D vectors (or a matrix `a` vs vector `b`)."""
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.ndim == 1:
        return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
    # a: (n, d) matrix, b: (d,) query -> (n,) similarities
    return (a @ b) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b))


# A toy 4-D "meaning space". Axes are made up but interpretable:
#   [is_royal, is_male, is_young, is_animal]
vocab = {
    "king":   [0.9, 0.9, 0.2, 0.0],
    "queen":  [0.9, 0.1, 0.2, 0.0],
    "man":    [0.1, 0.9, 0.4, 0.0],
    "woman":  [0.1, 0.1, 0.4, 0.0],
    "prince": [0.8, 0.9, 0.9, 0.0],
    "puppy":  [0.0, 0.5, 0.9, 1.0],
    "dog":    [0.0, 0.5, 0.3, 1.0],
}
words = list(vocab)
E = np.array([vocab[w] for w in words], dtype=np.float32)
print("embedding matrix:", E.shape, "(", len(words), "words x", E.shape[1], "dims )")

numpy 2.5.0
embedding matrix: (7, 4) ( 7 words x 4 dims )


## 5. Worked Examples

### Example 1 — Similarity is the whole point

Embeddings are only useful through a **similarity function**. Below we compute cosine similarity of every word against `king` and rank them. Watch `queen` and `prince` rise to the top while `dog`/`puppy` sink — the geometry recovers the semantics we baked into the axes.

In [2]:
query = "king"
sims = cosine(E, np.array(vocab[query], dtype=np.float32))

ranked = sorted(zip(words, sims), key=lambda t: -t[1])
print(f"cosine similarity to '{query}':\n")
for w, s in ranked:
    bar = "#" * int(round(s * 30))
    print(f"  {w:7s} {s:+.3f}  {bar}")

# Cosine ignores magnitude: scaling a vector by 10x leaves similarity unchanged.
scaled = 10.0 * np.array(vocab["queen"], dtype=np.float32)
print("\ncos(king, queen)       =", round(cosine(vocab["king"], vocab["queen"]), 3))
print("cos(king, 10x*queen)   =", round(cosine(vocab["king"], scaled), 3),
      "  <- identical: direction, not length, carries meaning")

cosine similarity to 'king':

  king    +1.000  ##############################
  prince  +0.883  ##########################
  queen   +0.787  ########################
  man     +0.768  #######################
  woman   +0.476  ##############
  dog     +0.342  ##########
  puppy   +0.341  ##########

cos(king, queen)       = 0.787
cos(king, 10x*queen)   = 0.787   <- identical: direction, not length, carries meaning


### Example 2 — Directions encode relationships (analogy arithmetic) + why we normalize

Two ideas in one cell:

1. **Vector arithmetic.** `king − man + woman` should land near `queen`. We compute that target vector and find its nearest neighbor in the vocab.
2. **Normalization.** We L2-normalize the matrix and show that on unit vectors **dot product equals cosine** — the reason real pipelines normalize once, store unit vectors, and then use the cheaper dot product (and why FAISS cosine search is "normalize + inner product"). See [[faiss]].

In [3]:
# --- 1. Analogy: king - man + woman ~= queen ---
target = (np.array(vocab["king"]) - np.array(vocab["man"]) + np.array(vocab["woman"]))
target = target.astype(np.float32)

sims = cosine(E, target)
ranked = sorted(zip(words, sims), key=lambda t: -t[1])
print("king - man + woman is closest to:")
for w, s in ranked[:3]:
    print(f"  {w:7s} {s:+.3f}")

# --- 2. On normalized vectors, dot product == cosine ---
norms = np.linalg.norm(E, axis=1, keepdims=True)
E_unit = E / norms                      # every row now has length 1
i, j = words.index("king"), words.index("queen")

dot_raw  = float(E[i] @ E[j])
dot_unit = float(E_unit[i] @ E_unit[j])
cos_val  = cosine(E[i], E[j])
print("\nraw dot(king, queen)       =", round(dot_raw, 3), " (depends on magnitude)")
print("unit dot(king, queen)      =", round(dot_unit, 3))
print("cosine(king, queen)        =", round(cos_val, 3),
      "  <- unit dot == cosine")

king - man + woman is closest to:
  queen   +1.000
  king    +0.787
  prince  +0.710

raw dot(king, queen)       = 0.94  (depends on magnitude)
unit dot(king, queen)      = 0.787
cosine(king, queen)        = 0.787   <- unit dot == cosine


### Example 3 — A miniature semantic search index (still pure numpy)

This is the pattern every vector database implements at scale: embed a corpus once, store the matrix, then for each query embed it and return the top-`k` by similarity. Here the "embedder" is a deterministic bag-of-characters hash so it runs with zero dependencies — a real system swaps in a learned model (next cell). The *retrieval loop is identical*.

In [4]:
def toy_embed(text, d=32):
    """Deterministic, dependency-free pseudo-embedding (hashed char trigrams).
    NOT semantic — just to demonstrate the index/search mechanics."""
    v = np.zeros(d, dtype=np.float32)
    t = f"  {text.lower()} "
    for i in range(len(t) - 2):
        v[hash(t[i:i+3]) % d] += 1.0
    n = np.linalg.norm(v)
    return v / n if n else v


corpus = [
    "how to reset your account password",
    "recover access to a locked account",
    "today's weather forecast and temperature",
    "best hiking trails in the mountains",
    "billing and subscription payment options",
]
index = np.stack([toy_embed(doc) for doc in corpus])   # (n_docs, d) — embed once

def search(query, k=2):
    q = toy_embed(query)
    scores = index @ q                                 # unit vectors -> dot == cosine
    top = np.argsort(-scores)[:k]
    return [(corpus[i], float(scores[i])) for i in top]

for q in ["unlock my login", "mountain walking routes"]:
    print(f"query: {q!r}")
    for doc, score in search(q):
        print(f"   {score:.3f}  {doc}")
    print()

query: 'unlock my login'
   0.603  best hiking trails in the mountains
   0.476  how to reset your account password

query: 'mountain walking routes'
   0.622  best hiking trails in the mountains
   0.475  recover access to a locked account



### Optional — real embeddings with a learned model (gated, no network unless present)

The cells above are intentionally synthetic so they always run. A real embedding model maps text into a space where *meaning*, not character overlap, determines closeness — so "unlock my login" matches "reset your password" even with **zero shared words**. This cell runs only if `sentence-transformers` is installed and you set `RUN_EMBEDDINGS=1`, so the notebook still executes top-to-bottom without it.

In [5]:
import importlib.util, os

if importlib.util.find_spec("sentence_transformers") and os.getenv("RUN_EMBEDDINGS"):
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")     # ~80 MB on first use, d=384
    docs = [
        "how to reset your account password",
        "best hiking trails in the mountains",
        "billing and subscription payment options",
    ]
    emb = model.encode(docs, normalize_embeddings=True)  # (3, 384) unit vectors
    q = model.encode(["I forgot my login credentials"], normalize_embeddings=True)
    scores = emb @ q[0]                                   # cosine via dot on unit vecs
    best = int(scores.argmax())
    print("embedding dim:", emb.shape[1])
    print("best match:", repr(docs[best]), "  score:", round(float(scores[best]), 3))
    print("(note: matched on meaning, not shared words)")
else:
    print("Skipped: `pip install sentence-transformers` and set RUN_EMBEDDINGS=1 to run.")
    print("Shape of the call: model.encode(docs, normalize_embeddings=True) -> (n, d) float array.")

Skipped: `pip install sentence-transformers` and set RUN_EMBEDDINGS=1 to run.
Shape of the call: model.encode(docs, normalize_embeddings=True) -> (n, d) float array.


## 6. Gotchas & Pitfalls

- **Never mix models in one index.** Vectors from `all-MiniLM-L6-v2` (d=384) and OpenAI `text-embedding-3-small` (d=1536) live in **different, incompatible spaces** — comparing them is meaningless even if you force the dimensions to match. Re-embed the whole corpus when you change models.
- **Normalize, or know why you didn't.** Most text models expect L2-normalized vectors and cosine/dot similarity. Skipping normalization lets a few high-magnitude vectors dominate rankings. Pick a metric and be consistent across indexing *and* querying.
- **Absolute scores are not calibrated.** Due to **anisotropy**, raw cosine scores often sit in a narrow band (e.g. everything 0.3–0.8). A 0.7 is not "70% relevant". Compare *relative* ranking and set thresholds empirically per model.
- **Embeddings ≠ keyword search.** They miss exact tokens — product codes, names, negation ("not covered"). For these, combine with BM25 (**hybrid search**) or you'll silently lose precision. See [[semantic-search]].
- **Chunking dominates RAG quality.** One embedding per giant document blurs meaning; too-small chunks lose context. Chunk size/overlap usually matters more than which model you pick. See [[rag]].
- **Context/token limits truncate silently.** Most embedding models cap input (e.g. 256–8192 tokens) and **silently truncate** the rest — your 50-page PDF became its first page. Chunk before embedding.
- **Dimensionality isn't free.** Bigger `d` means more RAM and slower search (often linearly). 384–768 dims is plenty for most tasks; reach for 1536+ only when you measure a quality win. Some models support **Matryoshka** truncation to trade dims for speed.
- **Staleness.** Embeddings are a snapshot of the model at embed time. Update the model → must re-embed everything. Budget for periodic re-indexing.
- **`hash()` is not an embedding.** (As in Example 3.) Real embeddings come from a *trained* model; character/hash tricks capture surface form, not meaning.

## 7. When to Use vs Alternatives

| Approach | What it captures | Use it when | Watch out for |
|---|---|---|---|
| **Dense embeddings** | Semantic similarity / meaning | Synonyms, paraphrase, fuzzy "about-ness", RAG | Misses exact tokens; scores uncalibrated; model dependency |
| **BM25 / TF-IDF (sparse lexical)** | Exact term overlap | Keywords, IDs, names, rare jargon, exact phrases | No synonymy; "car" ≠ "automobile" |
| **Hybrid (dense + sparse)** | Both meaning *and* exact terms | Production search/RAG — best of both, fused by RRF | More moving parts; need to tune fusion |
| **Cross-encoder / reranker** | Fine-grained query↔doc relevance | Re-ranking a top-k shortlist for precision | Too slow to score the whole corpus; pair with embeddings. See [[rerankers]] |
| **SQL `LIKE` / regex** | Literal substring match | Small data, exact patterns, no ML budget | No ranking, no semantics, doesn't scale to fuzzy |
| **Fine-tuned classifier** | A fixed label set | Stable categories, lots of labels, need calibrated probs | Rigid; retrain to add a class; not for open-ended retrieval |

**Rule of thumb:** embeddings are the right default for "find me things *about* X" at scale. For "find me the exact string X," use lexical. Most serious systems run **both** and fuse them, then optionally **rerank** the shortlist with a cross-encoder. The embedding model makes the points; a vector index ([[faiss]], [[pinecone]], [[chromadb]]) finds the nearest ones fast — choosing that index is a separate decision. See [[vector-db-comparison]].

## 8. Resources

- **Sentence-Transformers docs** (the de-facto open-source toolkit; pretrained models, pooling, similarity): https://www.sbert.net/
- **MTEB — Massive Text Embedding Benchmark** (the leaderboard for picking a model by task): https://huggingface.co/spaces/mteb/leaderboard
- **OpenAI embeddings guide** (API shape, dimensions, Matryoshka truncation): https://platform.openai.com/docs/guides/embeddings
- **Cohere — "What are embeddings?"** (clear, illustrated intuition): https://cohere.com/llmu/sentence-word-embeddings
- **Google ML Crash Course — Embeddings module** (foundations, how they're trained): https://developers.google.com/machine-learning/crash-course/embeddings
- **"The Illustrated Word2vec"** (Jay Alammar — where the king−man+woman intuition comes from): https://jalammar.github.io/illustrated-word2vec/